In [1]:
import json
import time
from collections import Counter, deque
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

In [2]:
START_URLS = {
    "rekrutacja": [
        "https://kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku-studia-i-stopnia/",
        "https://kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia/rekrutacja-krok-po-kroku-studia-ii-stopnia/",
        "https://kandydacipb.edu.pl/rekrutacja/",
        "https://kandydacipb.edu.pl/faq/",
    ],
    "studia": [
        "https://kandydacipb.edu.pl/kierunki-studiow/",
        "https://kandydacipb.edu.pl/studia-i-stopnia/",
        "https://kandydacipb.edu.pl/studia-ii-stopnia/",
    ],
    "kontakt": [
        "https://pb.edu.pl/kontakt/dane-teleadresowe/",
        "https://kandydacipb.edu.pl/kontakt/",
        "https://pb.edu.pl/dss/kontakt/",
    ],
    "stypendia": [
        "https://pb.edu.pl/studenci/sekcja-swiadczen-dla-studentow/",
    ],
    "akademik": [
        "https://pb.edu.pl/studenci/akademiki-pb/",
    ],
    "sprawy_studenckie": [
        "https://pb.edu.pl/studenci/",
        "https://pb.edu.pl/dss/",
        "https://pb.edu.pl/sok/",
    ],
}

CATEGORY_DEPTH = {
    "rekrutacja": 1,
    "studia": 1,
    "kontakt": 0,
    "stypendia": 1,
    "akademik": 0,
    "sprawy_studenckie": 1,
}

HEADERS = {"User-Agent": "StudentAssistantBot/1.0"}
ALLOWED_DOMAINS = {"pb.edu.pl", "kandydacipb.edu.pl"}


In [3]:
ALLOW_PATTERNS = {
    "rekrutacja": [
        "kandydacipb.edu.pl/rekrutacja",
        "kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku",
        "kandydacipb.edu.pl/faq",
    ],
    "studia": [
        "kandydacipb.edu.pl/kierunki-studiow",
        "kandydacipb.edu.pl/studia-i-stopnia",
        "kandydacipb.edu.pl/studia-ii-stopnia",
    ],
    "kontakt": [
        "pb.edu.pl/kontakt/dane-teleadresowe",
        "kandydacipb.edu.pl/kontakt",
        "pb.edu.pl/dss/kontakt",
    ],
    "stypendia": [
        "pb.edu.pl/studenci/sekcja-swiadczen-dla-studentow",
    ],
    "akademik": [
        "pb.edu.pl/studenci/akademiki-pb",
    ],
    "sprawy_studenckie": [
        "pb.edu.pl/studenci",
        "pb.edu.pl/dss",
        "pb.edu.pl/sok",
    ],
}

RECRUITMENT_URL_MARKERS = [
    "harmonogram-rekrutacji",
    "progi-punktowe",
    "limity-miejsc",
    "egzaminy-wstepne",
    "dokumenty-rekrutacyjne",
    "punktowane-przedmioty",
    "oplata-za-przeprowadzenie-rekrutacji",
    "wzor-rekrutacyjny",
    "badania-lekarskie",
]

def is_useful_url(url, category):
    url = url.lower()
    return any(pattern in url for pattern in ALLOW_PATTERNS.get(category, []))

def assign_category(category, url):
    if category == "studia" and any(marker in url.lower() for marker in RECRUITMENT_URL_MARKERS):
        return "rekrutacja"
    return category


In [4]:
SKIP_WORDS = [
    "aktualnosci", "news", "wydarzenia", "kalendarz", "/202",
    "deklaracja-dostepnosci", "ocena-parametryczna", "mikroposwiadczenia",
    "nostryfikacja", "cudzoziemcy", "etyka", "szkolenia-studentow",
    "wymiana-studencka", "legia-akademicka", "wzory-dyplomow",
    "tag", "category", "author", "page/",
    "nggallery", "galeria", "wp-content",
    "polityka-prywatnosci", "cookies",
    ".jpg", ".jpeg", ".png", ".webp", ".svg", ".pdf", ".doc", ".docx",
    "facebook", "instagram", "youtube", "linkedin",
    "mailto:", "tel:",
    "bialjam", "juwenalia", "festiwal", "konkurs",
    "wosp", "targi", "piknik", "konferencja",
    "kolo-naukowe", "samorzad", "erasmus", "mostech",
    "patent", "ekoskora", "rover", "zawod-inzynier",
    "obsluga-informatyczna", "faq-obsluga-informatyczna",
    "redakcja-serwisu-www", "mapa-social-media",
]

def should_skip(url):
    url = url.lower()
    return any(word in url for word in SKIP_WORDS)


In [5]:
BAD_MARKERS = [
    "Ustawienia ciasteczek",
    "Ta strona używa ciasteczek",
    "W ramach naszego serwisu www stosujemy pliki cookies",
    "Używamy plików cookie",
    "Zamknij ustawienia ciasteczek RODO",
    "Polityka prywatności Niezbędne pliki cookie",
    "Powered by Zgodności ciasteczek z RODO",
]

def remove_cookie_text(text):
    for marker in BAD_MARKERS:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
    return text.strip()

In [6]:
def normalize_url(url):
    parsed = urlparse(url)
    return parsed._replace(query="", fragment="").geturl().rstrip("/")

def is_valid_url(url):
    if "{" in url or "}" in url:
        return False

    parsed = urlparse(url)
    if parsed.scheme not in ["https", "http"]:
        return False

    domain = parsed.netloc.lower().removeprefix("www.")
    return domain in ALLOWED_DOMAINS


In [7]:
def clean_text(soup, url):
    candidates = soup.select("main, article, .entry-content, #content")
    content = max(
        candidates,
        key=lambda element: len(element.get_text(separator=" ", strip=True)),
        default=soup,
    )

    if url.rstrip("/") == "https://kandydacipb.edu.pl/rekrutacja":
        for article in content.select("article.et_pb_post, article.post"):
            article.decompose()

    for tag in content.select("script, style, nav, footer, header, form, aside, noscript"):
        tag.decompose()

    text = content.get_text(separator=" ")
    return " ".join(text.split())


In [8]:
def get_title(soup):
    if soup.title:
        return soup.title.get_text(strip=True)
    return ""

In [9]:
def get_links(soup, url):
    if url.rstrip("/") == "https://kandydacipb.edu.pl/rekrutacja":
        for article in soup.select("article.et_pb_post, article.post"):
            article.decompose()

    return [urljoin(url, a["href"]) for a in soup.find_all("a", href=True)]


In [10]:
def scrape_page(url):
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    soup_for_links = BeautifulSoup(response.text, "html.parser")

    title = get_title(soup)
    links = get_links(soup_for_links, response.url)
    text = remove_cookie_text(clean_text(soup, response.url))

    return {
        "url": normalize_url(response.url),
        "title": title,
        "text": text,
        "links": links,
    }


In [11]:
def crawl_category(category, start_urls, max_depth=1):
    visited = set()
    results = []
    errors = []
    queue = deque((normalize_url(url), 0) for url in start_urls)

    while queue:
        url, depth = queue.popleft()

        if url in visited or depth > max_depth:
            continue

        visited.add(url)

        try:
            page = scrape_page(url)
        except Exception as error:
            errors.append({"url": url, "error": str(error)})
            continue

        if len(page["text"]) > 300:
            results.append({
                "url": page["url"],
                "title": page["title"],
                "text": page["text"],
                "category": assign_category(category, page["url"]),
            })

        for link in page["links"]:
            link = normalize_url(link)

            if should_skip(link) or not is_valid_url(link):
                continue
            if not is_useful_url(link, category):
                continue
            if link not in visited:
                queue.append((link, depth + 1))

        time.sleep(0.5)

    return results, errors


#### Scraping trwa około 1m30s.


In [ ]:
all_pages = []
scrape_errors = []

for category, urls in START_URLS.items():
    pages, errors = crawl_category(
        category,
        urls,
        max_depth=CATEGORY_DEPTH.get(category, 0),
    )
    all_pages.extend(pages)
    scrape_errors.extend(errors)

unique_pages = {}
for page in all_pages:
    url = normalize_url(page["url"])
    if url not in unique_pages:
        page["url"] = url
        unique_pages[url] = page

all_pages = list(unique_pages.values())

with open("data/pages.jsonl", "w", encoding="utf-8") as file:
    for page in all_pages:
        file.write(json.dumps(page, ensure_ascii=False) + "\n")

category_counts = Counter(page["category"] for page in all_pages)
print(f"Zapisano stron łącznie: {len(all_pages)}")
for category in START_URLS:
    category_pages = [page for page in all_pages if page["category"] == category]
    lengths = [len(page["text"]) for page in category_pages]
    if lengths:
        print(
            f"{category:20} strony={len(lengths):3} "
            f"znaki={sum(lengths):7} min={min(lengths):5} max={max(lengths):6}"
        )
    else:
        print(f"{category:20} strony=  0")

print(f"Błędy pobierania: {len(scrape_errors)}")
for error in scrape_errors:
    print(f"- {error['url']} -> {error['error']}")


Zapisano stron ??cznie: 48
rekrutacja           strony= 31 znaki=  98624 min=  524 max= 18127
studia               strony=  3 znaki=   1901 min=  321 max=  1035
kontakt              strony=  3 znaki=   1797 min=  370 max=   764
stypendia            strony=  4 znaki=  11858 min=  384 max=  3971
akademik             strony=  1 znaki=   2150 min= 2150 max=  2150
sprawy_studenckie    strony=  6 znaki=   7821 min=  326 max=  4972
B??dy pobierania: 0


In [13]:
for page in sorted(all_pages, key=lambda item: (item["category"], item["url"])):
    print(f"{page['category']:20} {page['url']}")


akademik             https://pb.edu.pl/studenci/akademiki-pb
kontakt              https://kandydacipb.edu.pl/kontakt
kontakt              https://pb.edu.pl/dss/kontakt
kontakt              https://pb.edu.pl/kontakt/dane-teleadresowe
rekrutacja           https://kandydacipb.edu.pl/faq
rekrutacja           https://kandydacipb.edu.pl/rekrutacja
rekrutacja           https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie
rekrutacja           https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/dokumenty-do-pobrania
rekrutacja           https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/odplatnosc
rekrutacja           https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/potwierdzanie-efektow-krok-po-kroku
rekrutacja           https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/warunki-rekrutacji
rekrutacja           https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/wykaz-kier